# Road AI - YOLOv8-seg Training Notebook
## Instance Segmentation for Road Defects & Infrastructure

This notebook trains a YOLOv8s-seg model on the Road AI dataset.

**14 Classes:**
- Defects: Pothole, Edge_Deterioration, Alligator_Cracking, Rutting, Subsidence, Cracking, Bleeding, Fretting, Repaired_Pothole
- Infrastructure: Line_Marking, Storm_Drain, Manhole, Guard_Rail, Traffic_Sign

### Setup:
1. Upload `road_ai_seg_dataset.zip` as a Kaggle Dataset
2. Add it to this notebook (Add Data -> Your Datasets)
3. Turn on GPU (Settings -> Accelerator -> GPU T4 x2)
4. Run All cells

In [ ]:
# Install ultralytics
!pip install ultralytics -q

In [ ]:
import os
import zipfile
import yaml
import torch
from pathlib import Path

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

In [ ]:
# Extract dataset
# UPDATE THIS PATH if your Kaggle dataset has a different name
KAGGLE_DATASET_PATH = '/kaggle/input'  # Kaggle mounts datasets here

# Find the zip file
zip_path = None
for root, dirs, files in os.walk(KAGGLE_DATASET_PATH):
    for f in files:
        if f.endswith('.zip'):
            zip_path = os.path.join(root, f)
            break
    # Also check if data is already extracted (no zip)
    if not zip_path:
        for d in dirs:
            if os.path.exists(os.path.join(root, d, 'data.yaml')):
                zip_path = 'ALREADY_EXTRACTED'
                DATASET_DIR = os.path.join(root, d)
                break
    if zip_path:
        break

DATASET_DIR = '/kaggle/working/dataset'

if zip_path and zip_path != 'ALREADY_EXTRACTED':
    print(f'Found ZIP: {zip_path}')
    print('Extracting...')
    os.makedirs(DATASET_DIR, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATASET_DIR)
    print('Extraction complete!')
elif zip_path == 'ALREADY_EXTRACTED':
    print(f'Dataset already extracted at: {DATASET_DIR}')
else:
    # Check if files are directly in input folder
    for root, dirs, files in os.walk(KAGGLE_DATASET_PATH):
        if 'data.yaml' in files:
            DATASET_DIR = root
            print(f'Dataset found at: {DATASET_DIR}')
            break
        if 'images' in dirs and 'labels' in dirs:
            DATASET_DIR = root
            print(f'Dataset found at: {DATASET_DIR}')
            break

# Count images
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DATASET_DIR, 'images', split)
    if os.path.exists(img_dir):
        count = len(os.listdir(img_dir))
        print(f'  {split}: {count} images')

In [ ]:
# Create/update data.yaml with correct paths
CLASS_NAMES = {
    0: 'Pothole',
    1: 'Edge_Deterioration',
    2: 'Alligator_Cracking',
    3: 'Rutting',
    4: 'Subsidence',
    5: 'Cracking',
    6: 'Bleeding',
    7: 'Fretting',
    8: 'Repaired_Pothole',
    9: 'Line_Marking',
    10: 'Storm_Drain',
    11: 'Manhole',
    12: 'Guard_Rail',
    13: 'Traffic_Sign'
}

data_yaml = {
    'path': DATASET_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 14,
    'names': CLASS_NAMES
}

yaml_path = os.path.join(DATASET_DIR, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f'data.yaml created at: {yaml_path}')
print(f'Classes: {len(CLASS_NAMES)}')

In [ ]:
# Train YOLOv8s-seg model
from ultralytics import YOLO

# Load pretrained YOLOv8s-seg model
model = YOLO('yolov8s-seg.pt')

# Train
results = model.train(
    data=yaml_path,
    epochs=50,             # 50 epochs (increase to 100 for better results)
    imgsz=640,             # Image size
    batch=16,              # Batch size (reduce to 8 if GPU memory error)
    device=0,              # GPU 0
    patience=15,           # Early stopping patience
    save=True,             # Save checkpoints
    save_period=10,        # Save every 10 epochs
    plots=True,            # Generate training plots
    
    # Data augmentation
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    
    # Optimizer
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    
    # Project
    project='/kaggle/working/road_ai',
    name='yolov8s-seg-road',
    exist_ok=True,
)

print('\nTraining complete!')

In [ ]:
# Validate the model
metrics = model.val()
print(f'\nmAP50: {metrics.seg.map50:.4f}')
print(f'mAP50-95: {metrics.seg.map:.4f}')

In [ ]:
# Test on sample images
import glob
test_images = glob.glob(os.path.join(DATASET_DIR, 'images/test/*'))[:5]

if test_images:
    results = model.predict(test_images, save=True, conf=0.25)
    print(f'Predictions saved! Check /kaggle/working/road_ai/yolov8s-seg-road/')
    
    for r in results:
        if r.masks is not None:
            print(f'  {Path(r.path).name}: {len(r.masks)} detections')
        else:
            print(f'  {Path(r.path).name}: No detections')

In [ ]:
# Copy best.pt to output for download
import shutil

best_pt = '/kaggle/working/road_ai/yolov8s-seg-road/weights/best.pt'
last_pt = '/kaggle/working/road_ai/yolov8s-seg-road/weights/last.pt'

# Copy to /kaggle/working for easy download
if os.path.exists(best_pt):
    shutil.copy2(best_pt, '/kaggle/working/best.pt')
    size_mb = os.path.getsize(best_pt) / (1024**2)
    print(f'best.pt copied to /kaggle/working/best.pt ({size_mb:.1f} MB)')
    print('Download this file and place it at:')
    print('  D:/Road_AI_Project/ai_models/weights/best.pt')

if os.path.exists(last_pt):
    shutil.copy2(last_pt, '/kaggle/working/last.pt')
    print(f'last.pt also copied')

print('\nDONE! Download best.pt from the Output tab (right side)')